# Vanke Data Confirmation Checker

Run this notebook after the date-range demo. It combines Temporary Input, provisional DTD and automated QC in one review table. No release occurs until the Checker enters an explicit decision.

In [ ]:
from pathlib import Path

import pandas as pd

from vanke_data_confirm_checker import build_checker_table, confirm_dates

PROJECT_DIR = Path.cwd().resolve()
WORKSPACE_DIR = PROJECT_DIR / "demo_workspace"
if not (WORKSPACE_DIR / "vanke_dtd_temporary_data.xlsx").exists():
    raise FileNotFoundError("Run vanke_pipeline_demo.ipynb first.")

## 1 Review Temporary data and QC

`Automated_QC = PASS` means the row has acceptable input QC, a provisional DTD and no unresolved exception. A row requiring Marker review cannot use the normal Checker release path.

In [ ]:
checker_table = build_checker_table(WORKSPACE_DIR)
display(checker_table)

## 2 Enter the human decision

Review the table first. Then enter the Checker name, select dates and change the decision to `APPROVE` or `REJECT`. Keep `PENDING` when no write is intended.

In [ ]:
CHECKER_NAME = "Teacher"
DATES_TO_CONFIRM = checker_table.loc[checker_table["Production_Status"] == "PENDING_CHECKER", "Date"].astype(str).tolist()
CHECKER_DECISION = "PENDING"   # Change to APPROVE or REJECT after review.

In [ ]:
if CHECKER_DECISION == "PENDING":
    print("PENDING: no confirmed file was changed. Review the table and enter a decision.")
elif CHECKER_DECISION in {"APPROVE", "REJECT"}:
    checker_table = confirm_dates(
        WORKSPACE_DIR,
        DATES_TO_CONFIRM,
        checker_name=CHECKER_NAME,
        decision=CHECKER_DECISION,
    )
    print(f"{CHECKER_DECISION}: decision recorded for {len(DATES_TO_CONFIRM)} date(s).")
    display(checker_table)
    print("\nLATEST CONFIRMED INPUT")
    display(pd.read_excel(WORKSPACE_DIR / "vanke.xlsx", sheet_name="Input").tail(10))
    print("\nLATEST CONFIRMED DTD")
    display(pd.read_excel(WORKSPACE_DIR / "vanke.xlsx", sheet_name="Output").tail(10))
else:
    raise ValueError("CHECKER_DECISION must be PENDING, APPROVE or REJECT.")